# Cut SMILES data and convert to sequences

In [ ]:
# python washsmi.py -i /home/yinhongyan/github/ProPepDesigner/data/process/GIPR_GLP1R_GCGR_summary_seq.csv -smi smiles -iso


In [ ]:
from rdkit import  Chem
from rdkit.Chem import Draw, rdmolops, AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
import pandas as pd
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
import sys
sys.path.append('../../myutils')
from smiles2seq import peptide_smiles2seq

In [ ]:
import pandas as pd
import numpy as np
import multiprocessing as mp
from functools import lru_cache
from itertools import chain
from collections import ChainMap


# smiles_column = 'Backbone_smiles'  
smiles_column = 'washsmi_Iso' 


test_df = pd.read_csv('../process/GIPR_GLP1R_GCGR_summary_seq.csv')
test_df['mol'] = test_df[smiles_column].map(Chem.MolFromSmiles)
df = test_df[test_df['mol'].notna()].reset_index(drop=True)


@lru_cache(maxsize=None)
def smiles2seq_cached(smiles: str):
    seq_ls, seq_smiles, seq_ls_raw, seq_smiles_raw = peptide_smiles2seq(smiles)
    pep_sequence_model = str(seq_ls)
    pep_sequence_smiles = str(seq_smiles)
    pep_sequence = '--'.join(seq_ls_raw)
    pep_smiles = str(seq_smiles_raw)
    length = len(seq_ls)
    nnaa_items = tuple((smi, seq) for seq, smi in zip(seq_ls_raw, seq_smiles_raw) if '*' in seq)
    return pep_sequence, pep_smiles, length, nnaa_items, pep_sequence_model, pep_sequence_smiles

def _worker(smiles: str):
    return smiles, smiles2seq_cached(smiles)

# 1) cut
ls_smis = df[smiles_column].tolist()
with mp.Pool(mp.cpu_count()-10) as pool:
    results = dict(pool.imap_unordered(_worker, ls_smis, chunksize=200))

# 2) 
mapped = df[smiles_column].map(results)
cols = ['pep_sequence', 'pep_smiles', 'length', '_nnaa_items', 'pep_sequence_model','pep_smiles_model']
df[cols] = pd.DataFrame(mapped.tolist(), index=df.index)

# 3) 
from collections import ChainMap
NNAA = dict(ChainMap(*[dict(x) for x in df['_nnaa_items'] if x]))
df.drop(columns=['_nnaa_items'], inplace=True)
del df['mol']


In [ ]:
df.to_csv('../process/GIPR_GLP1R_GCGR_summary_seq.csv', index=None)

## 可视化每个片段

In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import rdDepictor
from IPython.display import SVG

def draw_mol(smiles,size=(1000,600),legend=None, highlightAtoms=None, highlightBonds=None):
    mol = Chem.MolFromSmiles(smiles)
    AllChem.Compute2DCoords(mol)
    rdDepictor.SetPreferCoordGen(True)
    d2d = rdMolDraw2D.MolDraw2DSVG(size[0],size[1])
    d2d.drawOptions().addAtomIndices=True
    # d2d.drawOptions().setHighlightColour((0.5,.8,.7,.7))
    d2d.DrawMolecule(mol,legend=legend, highlightAtoms=highlightAtoms, highlightBonds=highlightBonds)
    d2d.FinishDrawing()
    return SVG(d2d.GetDrawingText())

# 司美格鲁肽31：H-{Aib}-EGTFTSDVSSYLEGQAA-{C18 diacid-γ-Glu-(AEEA)2-Lys}-​​​EFIAWLVRGRG
# 也叫 Fmoc-L-Lys[Oct-(otBu)-Glu-(otBu)-AEEA-AEEA]-OH
semaglutide = '	CC[C@@H]([C@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC(CNC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC([C@@H](NC(CNC([C@@H](NC(C(C)(NC([C@@H](N)CC1=CN=CN1)=O)C)=O)CCC(O)=O)=O)=O)[C@H](O)C)=O)CC2=CC=CC=C2)=O)[C@H](O)C)=O)CO)=O)CC(O)=O)=O)C(C)C)=O)CO)=O)CO)=O)CC3=CC=C(O)C=C3)=O)CC(C)C)=O)CCC(O)=O)=O)=O)CCC(N)=O)=O)C)=O)C)=O)CCCCNC(COCCOCCNC(COCCOCCNC(CC[C@@H](NC(CCCCCCCCCCCCCCCCC(O)=O)=O)C(O)=O)=O)=O)=O)=O)CCC(O)=O)=O)CC4=CC=CC=C4)=O)C(N[C@H](C(N[C@H](C(N[C@H](C(N[C@H](C(N[C@H](C(NCC(N[C@H](C(NCC(O)=O)=O)CCCNC(N)=N)=O)=O)CCCNC(N)=N)=O)C(C)C)=O)CC(C)C)=O)CC5=CNC6=CC=CC=C65)=O)C)=O)C'
semaglutide_side = 'CC(C)(C)OC(=O)CCCCCCCCCCCCCCCCC(=O)N[C@@H](CCC(=O)NCCOCCOCC(=O)NCCOCCOCC(=O)NCCCC[C@@H](C(=O)O)NC(=O)OCC1C2=CC=CC=C2C3=CC=CC=C13)C(=O)OC(C)(C)C'

# 替尔泊肽39: Y-{Aib}-EGTFTSDYSI-{Aib}-LDKIAQ-{C20 diacid-gamma-Glu-(AEEA)2-Lys}-AFVQWLIAGGPSSGAPPPS-NH2
# 也叫 Fmoc-Lys(tBuO-Ara-Glu(AEEA-AEEA)-OtBu)-OH
tirzepatide = 'CC[C@H](C)[C@@H](C(=O)N[C@@H](C)C(=O)N[C@@H](CCC(=O)N)C(=O)N[C@@H](CCCCNC(=O)COCCOCCNC(=O)COCCOCCNC(=O)CC[C@H](C(=O)O)NC(=O)CCCCCCCCCCCCCCCCCCC(=O)O)C(=O)N[C@@H](C)C(=O)N[C@@H](CC1=CC=CC=C1)C(=O)N[C@@H](C(C)C)C(=O)N[C@@H](CCC(=O)N)C(=O)N[C@@H](CC2=CNC3=CC=CC=C32)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H]([C@@H](C)CC)C(=O)N[C@@H](C)C(=O)NCC(=O)NCC(=O)N4CCC[C@H]4C(=O)N[C@@H](CO)C(=O)N[C@@H](CO)C(=O)NCC(=O)N[C@@H](C)C(=O)N5CCC[C@H]5C(=O)N6CCC[C@H]6C(=O)N7CCC[C@H]7C(=O)N[C@@H](CO)C(=O)N)NC(=O)[C@H](CCCCN)NC(=O)[C@H](CC(=O)O)NC(=O)[C@H](CC(C)C)NC(=O)C(C)(C)NC(=O)[C@H]([C@@H](C)CC)NC(=O)[C@H](CO)NC(=O)[C@H](CC8=CC=C(C=C8)O)NC(=O)[C@H](CC(=O)O)NC(=O)[C@H](CO)NC(=O)[C@H]([C@@H](C)O)NC(=O)[C@H](CC9=CC=CC=C9)NC(=O)[C@H]([C@@H](C)O)NC(=O)CNC(=O)[C@H](CCC(=O)O)NC(=O)C(C)(C)NC(=O)[C@H](CC1=CC=C(C=C1)O)N'
tirzepatide_side = 'CC(C)(C)OC(=O)CCCCCCCCCCCCCCCCCCC(=O)N[C@H](CCC(=O)NCCOCCOCC(=O)NCCOCCOCC(=O)NCCCC[C@@H](C(=O)O)NC(=O)OCC1C2=CC=CC=C2C3=CC=CC=C13)C(=O)OC(C)(C)C'

# 瑞他鲁肽39: Tyr-{Aib}-Gln-Gly-Thr-Phe-Thr-Ser-Asp-Tyr-Ser-Ile-{α-Me-Leu}-Leu-Asp-Lys-{diacid-C20-gamma-Glu-(AEEA)-Lys}-Ala-Gln-{Aib}-Ala-Phe-Ile-Glu-Tyr-Leu-Leu-Glu-Gly-Gly-Pro-Ser-Ser-Gly-Ala-Pro-Pro-Pro-Ser-NH2
# 也叫 Fmoc-L-Lys[C20-OtBu-Glu(OtBu)-AEEA]-OH
retatrutide = 'CC[C@H](C)C(C(=O)N[C@@H](CCC(=O)O)C(=O)N[C@@H](CC1=CC=C(C=C1)O)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCC(=O)O)C(=O)NCC(=O)NCC(=O)N2CCC[C@H]2C(=O)N[C@@H](CO)C(=O)N[C@@H](CO)C(=O)NCC(=O)N[C@@H](C)C(=O)N3CCC[C@H]3C(=O)N4CCC[C@H]4C(=O)N5CCC[C@H]5C(=O)N[C@@H](CO)C(=O)N)NC(=O)[C@H](CC6=CC=CC=C6)NC(=O)[C@H](C)NC(=O)C(C)(C)NC(=O)[C@H](CCC(=O)N)NC(=O)[C@H](C)NC(=O)[C@H](CCCCNC(=O)COCCOCCNC(=O)CC[C@@H](C(=O)O)NC(=O)CCCCCCCCCCCCCCCCCCC(=O)O)NC(=O)[C@H](CCCCN)NC(=O)[C@H](CC(=O)O)NC(=O)[C@H](CC(C)C)NC(=O)[C@@](C)(CC(C)C)NC(=O)C([C@@H](C)CC)NC(=O)[C@H](CO)NC(=O)[C@H](CC7=CC=C(C=C7)O)NC(=O)[C@H](CC(=O)O)NC(=O)[C@H](CO)NC(=O)[C@H]([C@@H](C)O)NC(=O)[C@H](CC8=CC=CC=C8)NC(=O)[C@H]([C@@H](C)O)NC(=O)CNC(=O)[C@H](CCC(=O)N)NC(=O)C(C)(C)NC(=O)[C@H](CC9=CC=C(C=C9)O)N'
## 对这个单独的侧链Pubchem上没有手性，但是瑞他鲁肽中是有手性的，所以按照瑞他鲁肽中的确定结构
retatrutide_side = 'CC(C)(C)OC(CCCCCCCCCCCCCCCCCCC(NC(CCC(NCCOCCOCC(NCCCCC(C(O)=O)NC(OCC1c(cccc2)c2-c2c1cccc2)=O)=O)=O)C(OC(C)(C)C)=O)=O)=O'

# 利拉鲁肽31: HAEGTFTSDVSSYLEGQAA-{Lys-N6-[N-(1-oxohexadecyl)-L-g-glutamyl]}-EFIAWLVRGRG
# 也叫 Pal-Glu(OSu)-OH
Liraglutide = 'CCCCCCCCCCCCCCCC(=O)N[C@@H](CCC(=O)NCCCC[C@@H](C(=O)N[C@@H](CCC(=O)O)C(=O)N[C@@H](CC1=CC=CC=C1)C(=O)N[C@@H]([C@@H](C)CC)C(=O)N[C@@H](C)C(=O)N[C@@H](CC2=CNC3=CC=CC=C32)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](C(C)C)C(=O)N[C@@H](CCCNC(=N)N)C(=O)NCC(=O)N[C@@H](CCCNC(=N)N)C(=O)NCC(=O)O)NC(=O)[C@H](C)NC(=O)[C@H](C)NC(=O)[C@H](CCC(=O)N)NC(=O)CNC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC4=CC=C(C=C4)O)NC(=O)[C@H](CO)NC(=O)[C@H](CO)NC(=O)[C@H](C(C)C)NC(=O)[C@H](CC(=O)O)NC(=O)[C@H](CO)NC(=O)[C@H]([C@@H](C)O)NC(=O)[C@H](CC5=CC=CC=C5)NC(=O)[C@H]([C@@H](C)O)NC(=O)CNC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](C)NC(=O)[C@H](CC6=CN=CN6)N)C(=O)O'
Liraglutide_side = 'CCCCCCCCCCCCCCCC(=O)N[C@@H](CCC(=O)ON1C(=O)CCC1=O)C(=O)O'

In [ ]:
draw_mol(retatrutide_side, size=(3000,500), legend='Liraglutide-{Lys-N6-[N-(1-oxohexadecyl)-L-g-glutamyl]}', highlightAtoms=None, highlightBonds=None)
